In [ ]:
!pip install pypdf tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 6.3 MB/s eta 0:00:00


In [ ]:
import os
import json
from pypdf import PdfReader
from tqdm import tqdm


In [ ]:
!unzip /content/pdfs.zip -d pdfs


Archive:  /content/pdfs.zip
   creating: pdfs/pdfs/
  inflating: pdfs/pdfs/3. Financing Schemes for Sustainable Development - DOMAIN BASED.pdf  
  inflating: pdfs/pdfs/e-Book-Startup-India-Seed-Fund-Scheme-2 - DATE ALSO YES.pdf  
  inflating: pdfs/pdfs/ffaf37a755b0218b3adfbbf7daa4ab52 - yess.pdf  
  inflating: pdfs/pdfs/Karnataka_State_Report_26-05-2022.pdf  
  inflating: pdfs/pdfs/Kerala_State_Report_03-06-2022.pdf  
  inflating: pdfs/pdfs/MSME_Schemes_English_0 - YES.pdf  
  inflating: pdfs/pdfs/NASSCOM Zinnov Indian Tech Startup Report 2024 - YES.pdf  
  inflating: pdfs/pdfs/Scheme-booklet-Eng - YES(25).pdf  
  inflating: pdfs/pdfs/Startup India Kit_v5 -YES.pdf  
  inflating: pdfs/pdfs/Startup Policy.pdf  
  inflating: pdfs/pdfs/Startup-Playbook-DPIIT-Recognised-Startups-in-India - YESSSS (25).pdf  
  inflating: pdfs/pdfs/startup_funding.csv  
  inflating: pdfs/pdfs/Tamil_Nadu_State_Report_03-06-2022.pdf  


In [ ]:
PDF_DIR = "/content/pdfs/pdfs"
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]

print("✅ PDFs found:", len(pdf_files))
for f in pdf_files:
    print("-", f)

✅ PDFs found: 12
- Scheme-booklet-Eng - YES(25).pdf
- 3. Financing Schemes for Sustainable Development - DOMAIN BASED.pdf
- Kerala_State_Report_03-06-2022.pdf
- e-Book-Startup-India-Seed-Fund-Scheme-2 - DATE ALSO YES.pdf
- NASSCOM Zinnov Indian Tech Startup Report 2024 - YES.pdf
- ffaf37a755b0218b3adfbbf7daa4ab52 - yess.pdf
- Karnataka_State_Report_26-05-2022.pdf
- Startup India Kit_v5 -YES.pdf
- Startup-Playbook-DPIIT-Recognised-Startups-in-India - YESSSS (25).pdf
- MSME_Schemes_English_0 - YES.pdf
- Startup Policy.pdf
- Tamil_Nadu_State_Report_03-06-2022.pdf


In [ ]:
CORE_FUNDING_DOCS = [
    "e-Book-Startup-India-Seed-Fund-Scheme-2.pdf",
    "Scheme-booklet-Eng.pdf",
    "3. Financing Schemes for Sustainable Development.pdf"
]

ELIGIBILITY_DOCS = [
    "Startup India Kit_v5.pdf",
    "Startup-Playbook-DPIIT-Recognised-Startups-in-India.pdf",
    "ffaf37a755b0218b3adfbbf7daa4ab52.pdf"
]

ECOSYSTEM_DOCS = [
    "NASSCOM Zinnov Indian Tech Startup Report 2024.pdf"
]

STATE_DOCS = [
    "Karnataka_State_Report_26-05-2022.pdf",
    "Tamil_Nadu_State_Report_03-06-2022.pdf",
    "Kerala_State_Report_03-06-2022.pdf",
    "Telangana_State_Report_10-05-2022(1).pdf"
]


In [ ]:
def get_layer(pdf_name):
    name = pdf_name.lower()

    if "seed" in name or "scheme-booklet" in name or "msme" in name or "sidbi" in name:
        return "core_funding"

    elif "playbook" in name or "startup india kit" in name or "promotion" in name:
        return "eligibility_guidance"

    elif "nasscom" in name or "zinnov" in name:
        return "funding_trends"

    elif "karnataka" in name or "tamil" in name or "kerala" in name or "telangana" in name:
        return "state_level"

    else:
        return "unknown"


def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text
import re

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('\x0c', '')
    return text.strip()
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap

    return chunks
import json
from tqdm import tqdm

all_chunks = []

for pdf_name in tqdm(pdf_files):
    pdf_path = os.path.join(PDF_DIR, pdf_name)

    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)
    chunks = chunk_text(cleaned_text)

    layer = get_layer(pdf_name)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "metadata": {
                "source_file": pdf_name,
                "chunk_id": i,
                "layer": layer
            }
        })

print("✅ Total chunks created:", len(all_chunks))

100%|██████████| 12/12 [00:20<00:00,  1.72s/it]

✅ Total chunks created: 243


In [ ]:
OUTPUT_FILE = "funding_chunks_layered.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"✅ Saved to {OUTPUT_FILE}")

✅ Saved to funding_chunks_layered.json
